[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P19/P19-1_Outil_Modele.ipynb)

# Du script au premier modèle

Chapitre 19 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Votre responsable veut une règle qui prédise la durée de vie d'un organe neuf, avec une erreur chiffrée et un outil que tout collègue peut rejouer.

**Objectif terminal.** Ce chapitre montre comment écrire cet outil, le tester, puis juger un premier modèle.

**Objectif instrumental.**

- Écrire un module testé et versionné, puis traiter un lot de classeurs.
- Juger un modèle linéaire sur une base de test, puis mener un projet complet.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Les chapitres 3, 7, 15 et 18, après une filière scientifique.

## Source de données

| Fichier | Lignes | Colonnes | Unités |
|:-|-:|-:|:-|
| `MecaIA-RUL.xlsx` | 240 | 6 | couple en N·m, puissance en kW, masse en kg, temps de réponse en ms, durée de vie résiduelle en jours |
| `MecaIA-Surveillance_Vibratoire.xlsx` | 150 | 25 | vitesse en tr/min, charge en kN, pression en bar, débit en L/min, vibrations en mm/s, température en °C, émission acoustique en dB, courant en A, puissance en kW, durée de vie en h |
| `MecaIA-Traitement_Acier.xlsx` | 150 | 6 | température en °C, temps en min, teneurs en %, résistance en MPa |
| `MecaIA-Suivi_Production_Machines.xlsx` | 20000 | 14 | ancienneté en années, température en °C, vibrations en mm/s, consommation en kWh/h, productivité en pièces/h, indice de santé de 0 à 100 |
| `MecaIA-Plaque_Trouee_FEA.xlsx` | 100 | 12 | dimensions en mm, contrainte en MPa, déplacement en mm, durée de calcul en s |
| `MecaIA-TimeSeries_RUL.xlsx` | 18855 | 10 | cycles en heures, charge en %, vitesse en tr/min, température en K, pression en bar, vibration en mm/s, courant en A, contamination en ppm |

Les classeurs sont lus dans le dossier `donnees` du dépôt `Meca-PY_Ed2026` ; ouvert seul, par exemple dans Colab, le notebook les télécharge une fois depuis le dépôt public.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 19.1 | Combien de fonctions passent leurs exemples ? | 15 min |
| 19.2 | Traiter un dossier de classeurs | 20 min |
| 19.3 | Les tests unitaires du module | 30 min |
| 19.4 | Lire, nettoyer et apprendre une droite | 15 min |
| 19.5 | Noter une droite sur la base de test | 15 min |
| 19.6 | Un script complet pour la durée de vie | 20 min |
| 19.7 | Des fenêtres et des images pour un réseau (défi) | 35 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### Lecture des jeux

Les jeux se cherchent d'abord dans le dépôt cloné, à deux dossiers au-dessus de ce notebook.

In [ ]:
from pathlib import Path
import urllib.request

# Les jeux du chapitre : nom du classeur et adresse de la copie publique.
JEUX = {
    "MecaIA-RUL.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-RUL.xlsx",
    "MecaIA-TimeSeries_RUL.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-TimeSeries_RUL.xlsx",
    "MecaIA-Surveillance_Vibratoire.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-Surveillance_Vibratoire.xlsx",
    "MecaIA-Traitement_Acier.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-Traitement_Acier.xlsx",
    "MecaIA-Suivi_Production_Machines.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-Suivi_Production_Machines.xlsx",
    "MecaIA-Plaque_Trouee_FEA.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-Plaque_Trouee_FEA.xlsx",
}

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-PY_Ed2026", "donnees")]
dossier = next((d for d in candidats if all((d / f).is_file() for f in JEUX)), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique des jeux depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    for fichier, adresse in JEUX.items():
        if not (dossier / fichier).is_file():
            try:
                urllib.request.urlretrieve(adresse, dossier / fichier)
            except OSError as erreur:
                raise FileNotFoundError(
                    "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
DONNEES = dossier
print("jeux lus dans le dossier des données :", ", ".join(JEUX))

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [ ]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))
print("pandas", version("pandas"))

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook lit les jeux de sa première cellule et reproduit le code des modules cités ; les téléverser sert à les importer dans vos propres cellules.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = list(JEUX)                                  # les classeurs de la première cellule
MODULES_DU_LIVRE = ["meca.py"]                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Le module du livre

Le chapitre étudie le module `meca.py` et sa suite de tests `test_meca.py`. Leur texte complet est donné ci-dessous ; la cellule qui suit les écrit dans un dossier temporaire et les importe, comme le ferait un module posé à côté du notebook.

`meca.py`

In [ ]:
TEXTE_MECA = r'''# SPDX-License-Identifier: MIT
"""meca : le module de calcul de l'ingénieur, fil rouge du livre.

Chaque fonction est courte, documentée, et porte son test dans sa docstring (module doctest).
Pour les rejouer tous :

    python meca.py            # affiche « meca : N fonctions, 0 échec »

Le module se lit dans l'ordre du livre :

    1. calculs de résistance des matériaux (chapitre 3) et Monte-Carlo (chapitre 4) ;
    2. outils de l'auteur pour mettre à l'échelle, borner, tracer une droite (chapitres 5 et 19) ;
    3. lecture, contrôle et horodatage des classeurs de mesures (chapitres 7 à 9) ;
    4. prétraitement et évaluation en quatre temps, noms de l'auteur (chapitre 19) ;
    5. données prêtes pour les réseaux de neurones (chapitre 19).

Les noms anglais `preprocessing_*` et `evaluation_*` sont ceux de l'auteur : ils sont gardés tels quels.
"""
import math

import numpy as np
import pandas as pd

__version__ = "1.0.0"

# Les 15 fonctions reprises du code de l'auteur, sous leur nom.
FONCTIONS_AUTEUR = (
    "preprocessing_imputation", "preprocessing_outliers", "preprocessing_Xy", "preprocessing",
    "evaluation_perf", "evaluation_lc", "model_equation",
    "scale_to_range", "clip_round", "safe_div", "minmax_scale_01", "_format_unique_values",
    "fit_line", "eq_label", "fmt_num",
)


# Partie 1. Résistance des matériaux et Monte-Carlo (chapitres 3 et 4)

def contrainte_MPa(F_N, d_mm):
    """Contrainte normale dans une tige ronde, en MPa, pour un effort F en N et un diamètre d en mm.

    >>> round(contrainte_MPa(12000.0, 10.0), 2)
    152.79
    >>> contrainte_MPa(12000.0, 0.0)
    Traceback (most recent call last):
        ...
    ValueError: le diamètre doit être positif
    """
    if d_mm <= 0:
        raise ValueError("le diamètre doit être positif")
    return F_N / (math.pi * d_mm**2 / 4)


def securite(Re_MPa, F_N, d_mm):
    """Coefficient de sécurité d'une tige : limite d'élasticité sur contrainte de service.

    >>> round(securite(355.0, 12000.0, 10.0), 2)
    2.32
    """
    return Re_MPa / contrainte_MPa(F_N, d_mm)


def kt_trou(d_mm, W_mm):
    """Kt d'une plaque de largeur W percée d'un trou central de diamètre d, en traction.

    Formule de Heywood : Kt = 2 + (1 - d/W)**3, rapportée à la contrainte nette F / ((W - d) e).
    Source : R. B. Heywood, Designing by Photoelasticity, Chapman and Hall, 1952 ; formule reprise
    par W. D. Pilkey et D. F. Pilkey, Peterson's Stress Concentration Factors, Wiley, et par
    Roark's Formulas for Stress and Strain, McGraw-Hill. Domaine : 0 < d/W < 1.

    >>> round(kt_trou(10.0, 50.0), 3)
    2.512
    >>> kt_trou(60.0, 50.0) != kt_trou(60.0, 50.0)     # nan hors du domaine
    True
    """
    rapport = d_mm / W_mm
    if rapport <= 0 or rapport >= 1:
        return float("nan")      # hors du domaine : pas de trou, ou plus de matière
    return 2 + (1 - rapport)**3


def moment_quadratique_mm4(b_mm, h_mm):
    """Moment quadratique d'une section rectangulaire : b h³ / 12, en mm⁴.

    >>> moment_quadratique_mm4(20.0, 40.0)
    106666.66666666667
    """
    return b_mm * h_mm**3 / 12


def fleche_mm(F_N, L_mm, b_mm, h_mm, E_MPa):
    """Flèche au centre, poutre sur deux appuis, charge F au milieu : F L³ / (48 E I).

    >>> round(fleche_mm(2000.0, 1000.0, 20.0, 40.0, 210000.0), 4)
    1.8601
    """
    return F_N * L_mm**3 / (48 * E_MPa * moment_quadratique_mm4(b_mm, h_mm))


def tau_MPa(d_mm, T_Nm):
    """Cisaillement maximal d'un arbre plein en torsion : 16 T / (pi d³), T en N.m, d en mm.

    >>> round(tau_MPa(34.881591, 500.0), 2)
    60.0
    """
    return 16 * T_Nm * 1000 / (math.pi * d_mm**3)


def dichotomie(g, a, b, tolerance=0.005):
    """Zéro de g dans [a, b] par dichotomie : rend (solution, nombre de tours).

    g doit changer de signe entre a et b. La boucle s'arrête quand l'encadrement fait moins que la tolérance.

    >>> x, tours = dichotomie(lambda d: tau_MPa(d, 500.0) - 60.0, 10.0, 100.0)
    >>> round(x, 2), tours
    (34.88, 15)
    >>> dichotomie(lambda x: x**2 + 1.0, 0.0, 1.0)
    Traceback (most recent call last):
        ...
    ValueError: g ne change pas de signe entre a et b
    """
    if g(a) * g(b) > 0:
        raise ValueError("g ne change pas de signe entre a et b")
    tours = 0
    while b - a > tolerance:
        milieu = (a + b) / 2
        if g(a) * g(milieu) <= 0:
            b = milieu           # le signe change dans la moitié gauche
        else:
            a = milieu           # il change dans la moitié droite
        tours = tours + 1
    return (a + b) / 2, tours


def jeu_monte_carlo(cotes, tirages, rng):
    """Jeu fonctionnel d'une chaîne de cotes, par tirage : somme signée de cotes de loi normale.

    `cotes` est une liste de (nominale, tolérance, signe). Chaque cote suit une loi normale d'écart-type
    tolérance / 3 (la tolérance couvre trois écarts-types) ; le signe vaut +1 si la cote agrandit le jeu,
    -1 si elle le réduit.

    >>> rng = np.random.default_rng(0)
    >>> jeu = jeu_monte_carlo([(100.0, 0.3, 1), (60.0, 0.15, -1), (38.0, 0.15, -1)], 20000, rng)
    >>> jeu.shape
    (20000,)
    >>> round(float(jeu.mean()), 2)
    2.0
    """
    jeu = np.zeros(tirages)
    for nominale, tolerance, signe in cotes:
        jeu = jeu + signe * rng.normal(nominale, tolerance / 3, tirages)
    return jeu


# Partie 2. Outils de l'auteur : mise à l'échelle, bornes, droite et légendes (chapitres 5 et 19)

def scale_to_range(series, new_min, new_max):
    """Met une série à l'échelle [new_min, new_max] ; une série constante rend le milieu de l'intervalle.

    >>> scale_to_range(pd.Series([2.0, 4.0, 6.0]), 0, 100).tolist()
    [0.0, 50.0, 100.0]
    >>> scale_to_range(pd.Series([5.0, 5.0]), 0, 10).tolist()
    [5.0, 5.0]
    """
    series = pd.to_numeric(series, errors="coerce")
    xmin = np.nanmin(series.values)
    xmax = np.nanmax(series.values)

    if not np.isfinite(xmin) or not np.isfinite(xmax) or np.isclose(xmin, xmax):
        return pd.Series(np.full(len(series), (new_min + new_max) / 2), index=series.index)

    return (series - xmin) / (xmax - xmin) * (new_max - new_min) + new_min


def clip_round(s: pd.Series, a: float, b: float, ndigits: int) -> pd.Series:
    """Borne une série entre a et b, puis l'arrondit à ndigits décimales.

    >>> clip_round(pd.Series([1.234, 5.678, 9.0]), 2.0, 6.0, 1).tolist()
    [2.0, 5.7, 6.0]
    """
    return s.clip(lower=a, upper=b).round(ndigits)


def safe_div(a, b, default=0.0):
    """Division protégée : un dénominateur nul rend `default` au lieu d'une erreur ou d'un infini.

    >>> safe_div(np.array([1.0, 2.0, 3.0]), np.array([0.0, 4.0, 2.0])).tolist()
    [0.0, 0.5, 1.5]
    """
    b_ = np.where(b == 0, np.nan, b)
    out = a / b_
    return np.nan_to_num(out, nan=default, posinf=default, neginf=default)


def minmax_scale_01(v):
    """Met un vecteur à l'échelle [0, 1] ; un vecteur constant rend 0,5.

    >>> minmax_scale_01([10.0, 15.0, 20.0]).tolist()
    [0.0, 0.5, 1.0]
    >>> minmax_scale_01([3.0, 3.0]).tolist()
    [0.5, 0.5]
    """
    v = np.asarray(v, dtype=float)
    vmin, vmax = np.min(v), np.max(v)
    if vmax - vmin <= 1e-12:
        return np.full_like(v, 0.5)
    return (v - vmin) / (vmax - vmin)


def _format_unique_values(series: pd.Series, max_values: int = 30) -> str:
    """Résumé lisible des valeurs uniques d'une colonne (avec troncature si la liste est longue).

    >>> print(_format_unique_values(pd.Series(["A", "B", "A", None])))
    n=4, n_nan=1, n_unique=2
    uniques (max 30) = [A, B]
    >>> print(_format_unique_values(pd.Series([1, 2, 3, 4]), max_values=2))
    n=4, n_nan=0, n_unique=4
    uniques (max 2) = [1, 2] ... (+2 autres)
    """
    s = series
    n_total = len(s)
    n_nan = int(s.isna().sum())
    n_unique = int(s.nunique(dropna=True))

    uniques = s.dropna().unique()

    uniques_str = [str(v) for v in uniques[:max_values]]
    suffix = ""
    if len(uniques) > max_values:
        suffix = f" ... (+{len(uniques) - max_values} autres)"

    return (
        f"n={n_total}, n_nan={n_nan}, n_unique={n_unique}\n"
        f"uniques (max {max_values}) = [{', '.join(uniques_str)}]{suffix}"
    )


def fit_line(x, y):
    """Droite des moindres carrés d'un nuage : rend (pente, ordonnée à l'origine).

    >>> m, b = fit_line([0.0, 1.0, 2.0, 3.0], [1.0, 3.0, 5.0, 7.0])
    >>> round(m, 6), round(b, 6)
    (2.0, 1.0)
    """
    m, b = np.polyfit(x, y, 1)
    return float(m), float(b)


def eq_label(m, b, decimals=2):
    """Équation de droite pour une légende de figure.

    >>> eq_label(2.0, 1.0)
    'y = 2.00X + 1.00'
    >>> eq_label(-0.1138, 358.0, decimals=3)
    'y = -0.114X + 358.000'
    """
    sign = "+" if b >= 0 else "-"
    return f"y = {m:.{decimals}f}X {sign} {abs(b):.{decimals}f}"


def fmt_num(v):
    """Format numérique pour une légende : notation scientifique hors de [0,001 ; 10 000[.

    >>> [fmt_num(v) for v in (0, 0.5, 1234.5, 123456.0, 0.0001)]
    ['0', '0.50', '1234.5', '1.23e+05', '1.00e-04']
    """
    if v == 0:
        return "0"
    av = abs(v)
    if av >= 1e4 or av < 1e-3:
        return f"{v:.2e}"
    if av >= 100:
        return f"{v:.1f}"
    return f"{v:.2f}"


# Partie 3. Lire, contrôler et horodater des mesures (chapitres 7 à 9)

def lire_classeur(chemin, **options):
    """Lit un classeur et garde l'original : rend (data, df), df étant une copie de travail.

    >>> import os, tempfile
    >>> dossier = tempfile.mkdtemp()
    >>> chemin = os.path.join(dossier, "mesures.xlsx")
    >>> pd.DataFrame({"x": [1, 2, 3]}).to_excel(chemin, index=False)
    >>> data, df = lire_classeur(chemin)
    >>> df.loc[0, "x"] = 99
    >>> int(data.loc[0, "x"]), int(df.loc[0, "x"])
    (1, 99)
    """
    data = pd.read_excel(chemin, **options)
    df = data.copy()
    return data, df


def taux_manquants(df):
    """Taux de valeurs manquantes par colonne, en pourcentage, trié par ordre décroissant.

    >>> d = pd.DataFrame({"a": [1.0, None, None, 4.0], "b": [1.0, 2.0, 3.0, None], "c": [1, 2, 3, 4]})
    >>> taux_manquants(d).round(1).to_dict()
    {'a': 50.0, 'b': 25.0, 'c': 0.0}
    """
    return (df.isna().mean() * 100).sort_values(ascending=False)


def cellules_et_lignes(df):
    """Cellules manquantes et lignes touchées : rend (cellules, lignes), deux comptes qui ne se confondent pas.

    >>> d = pd.DataFrame({"a": [1.0, None, None, 4.0], "b": [1.0, 2.0, 3.0, None]})
    >>> cellules_et_lignes(d)
    (3, 3)
    """
    return int(df.isna().sum().sum()), int(df.isna().any(axis=1).sum())


def verifier_bornes(df, bornes, arreter=True):
    """Compte, par colonne, les mesures hors de leur plage physique ; `arreter` lève une erreur au premier écart.

    `bornes` associe un nom de colonne à (minimum, maximum). Les valeurs manquantes ne sont pas des écarts.

    >>> d = pd.DataFrame({"T": [20.0, 25.0, 900.0], "P": [1.0, 2.0, 3.0]})
    >>> verifier_bornes(d, {"T": (0.0, 200.0), "P": (0.0, 10.0)}, arreter=False)
    {'T': 1, 'P': 0}
    >>> verifier_bornes(d, {"T": (0.0, 200.0)})
    Traceback (most recent call last):
        ...
    ValueError: T : 1 mesure(s) hors de [0.0, 200.0]
    """
    ecarts = {}
    for colonne, (mini, maxi) in bornes.items():
        hors = int((~df[colonne].between(mini, maxi) & df[colonne].notna()).sum())
        ecarts[colonne] = hors
        if hors and arreter:
            raise ValueError(f"{colonne} : {hors} mesure(s) hors de [{mini}, {maxi}]")
    return ecarts


def indexer_horodatage(df_in, depart, colonne="time", unite="D", arrondi="h", nom="Horodatage"):
    """Remplace une colonne de temps relatif par un index de dates (démarche de l'auteur).

    La colonne `colonne` compte le temps écoulé depuis `depart`, dans l'unité de la fiche du jeu ;
    l'index obtenu est arrondi à `arrondi` puis nommé `nom`.

    >>> d = pd.DataFrame({"time": [0.0, 0.1667, 0.3334], "T": [20.0, 21.0, 22.0]})
    >>> out = indexer_horodatage(d, pd.Timestamp("2025-01-01 08:00"))
    >>> [str(t) for t in out.index]
    ['2025-01-01 08:00:00', '2025-01-01 12:00:00', '2025-01-01 16:00:00']
    >>> out.index.name
    'Horodatage'
    """
    df = df_in.copy()
    df = df.set_index(colonne)
    df.index = depart + pd.to_timedelta(df.index, unit=unite)
    df.index = df.index.round(arrondi)
    df.index.name = nom
    return df


# Partie 4. Prétraitement et évaluation en quatre temps, noms de l'auteur (chapitre 19)

def preprocessing_imputation(df_in):
    """Prétraitement propre au jeu de la plaque trouée : colonnes utiles, section, contrainte nominale, Kt.

    Le jeu de l'auteur porte les colonnes de la simulation par éléments finis ; on garde d0 (diamètre du trou)
    et Kt = contrainte maximale / contrainte nominale, avec une force de 1000 N.

    >>> d = pd.DataFrame({"Unnamed: 0": [0], "L": [400], "h": [200], "k": [20], "x0": [200], "y0": [100],
    ...                   "d0": [10.0], "Material": ["Acier"], "Stress/MPa": [0.65789474],
    ...                   "Disp/mm": [0.0005], "Msg": [None], "FEA_Runtime": [10.0]})
    >>> out = preprocessing_imputation(d)
    >>> list(out.columns), round(float(out.loc[0, "Kt"]), 3)
    (['d0', 'Kt'], 2.5)
    """
    df = df_in.copy()  # On refait une copie
    # Elimination de colonnes
    df = df.drop(['Disp/mm', 'Msg', 'FEA_Runtime', 'Material', 'Unnamed: 0'], axis=1)

    # Renommer
    df.rename(columns={"k": "t"}, inplace=True)

    # Kt
    df["Sec"] = df["t"] * (df["h"] - df["d0"])
    df["StressNom"] = 1000 / df["Sec"]
    df["Kt"] = df["Stress/MPa"] / df["StressNom"]

    # Final avec d0 et Kt
    key_cols = ['d0', 'Kt']
    df_out = df[key_cols]
    return df_out


def preprocessing_outliers(df_in):
    """Retire les points aberrants repérés par une forêt d'isolement (1 % supposé).

    >>> rng = np.random.default_rng(1)
    >>> d = pd.DataFrame({"x": rng.normal(0, 1, 200), "y": rng.normal(0, 1, 200)})
    >>> d.loc[0] = [50.0, 50.0]
    >>> out = preprocessing_outliers(d)
    >>> len(d) - len(out), 0 in out.index
    (2, False)
    """
    from sklearn.ensemble import IsolationForest

    model_isoF = IsolationForest(contamination=0.01, random_state=42)
    pred = model_isoF.fit_predict(df_in)             # 1 = normal, -1 = anomalie

    df_in = df_in.assign(outlier=(pred == -1))
    df_in = df_in[df_in["outlier"] == False]

    df_out = df_in.drop("outlier", axis=1)

    return df_out


def preprocessing_Xy(df_in):
    """Sépare X (toutes les colonnes sauf la dernière) et y (la dernière colonne).

    >>> X, y = preprocessing_Xy(pd.DataFrame({"a": [1, 2], "b": [3, 4], "cible": [5, 6]}))
    >>> X.shape, y.shape, y.name
    ((2, 2), (2,), 'cible')
    """
    # X = toutes les colonnes sauf la dernière en DataFrame 2D
    X = df_in.iloc[:, :-1]
    # y = la dernière colonne en série 1D
    y = df_in.iloc[:, -1]
    return X, y


def preprocessing(df_in, ordre="auteur"):
    """Rejoue d'un coup tout le prétraitement : rend X_train, X_test, y_train, y_test.

    ordre="auteur" suit l'enchaînement de l'auteur : imputation, aberrants, X et y, découpage.
    ordre="sans_fuite" découpe avant de chercher les aberrants, qui ne sont retirés que de l'apprentissage :
    le test reste intact. Le jeu attendu est celui de preprocessing_imputation.

    >>> rng = np.random.default_rng(2)
    >>> d = pd.DataFrame({"Unnamed: 0": range(50), "L": 400, "h": 200, "k": 20, "x0": 200, "y0": 100,
    ...                   "d0": np.linspace(10, 100, 50), "Material": "Acier",
    ...                   "Stress/MPa": rng.uniform(0.5, 1.5, 50), "Disp/mm": 0.0, "Msg": None,
    ...                   "FEA_Runtime": 1.0})
    >>> X_train, X_test, y_train, y_test = preprocessing(d)
    >>> (len(X_train), len(X_test))
    (39, 10)
    >>> X_train, X_test, y_train, y_test = preprocessing(d, ordre="sans_fuite")
    >>> (len(X_train), len(X_test))
    (39, 10)
    """
    from sklearn.model_selection import train_test_split

    df_in = preprocessing_imputation(df_in)
    if ordre == "auteur":
        df_in = preprocessing_outliers(df_in)
        X, y = preprocessing_Xy(df_in)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    elif ordre == "sans_fuite":
        X, y = preprocessing_Xy(df_in)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
        X_train, y_train = preprocessing_Xy(preprocessing_outliers(pd.concat([X_train, y_train], axis=1)))
    else:
        raise ValueError("ordre inconnu : " + str(ordre))
    return X_train, X_test, y_train, y_test


def evaluation_perf(model, X_train, y_train, X_test, y_test, tol=0.05):
    """Pourcentage de prédictions de la base de test dans une tolérance relative `tol`.

    >>> from sklearn.linear_model import LinearRegression
    >>> X = pd.DataFrame({"x": [1.0, 2.0, 3.0, 4.0]})
    >>> y = pd.Series([2.0, 4.0, 6.0, 8.0])
    >>> reg = LinearRegression().fit(X, y)
    >>> float(evaluation_perf(reg, X, y, X, y, tol=0.01))
    100.0
    """
    y_pred = model.predict(X_test)
    rel_error = np.abs(y_pred - y_test) / np.abs(y_test)
    return 100 * np.mean(rel_error <= tol)


def evaluation_lc(model, X_train, y_train, tracer=True):
    """Courbe d'apprentissage (validation croisée à 5 plis, score R²) : rend les tailles et les deux scores moyens.

    >>> import matplotlib
    >>> matplotlib.use("Agg")
    >>> from sklearn.linear_model import LinearRegression
    >>> rng = np.random.default_rng(3)
    >>> X = pd.DataFrame({"x": rng.uniform(0, 10, 60)})
    >>> y = 2.0 * X["x"] + rng.normal(0, 0.5, 60)
    >>> tailles, entrainement, validation = evaluation_lc(LinearRegression(), X, y, tracer=False)
    >>> len(tailles), bool(validation[-1] > 0.9)
    (10, True)
    """
    from sklearn.model_selection import learning_curve

    train_sizes = np.linspace(0.1, 1.0, 10)
    sizes, train_scores, test_scores = learning_curve(
        model, X_train, y_train, train_sizes=train_sizes, cv=5, scoring="r2"
    )
    if tracer:
        import matplotlib.pyplot as plt

        plt.figure(figsize=(6, 4))
        plt.plot(sizes, np.mean(train_scores, axis=1), "o-", label="Train")
        plt.plot(sizes, np.mean(test_scores, axis=1), "o-", label="Validation")
        plt.xlabel("Taille de l'échantillon d'apprentissage")
        plt.ylabel("Score")
        plt.legend()
        plt.tight_layout()
    return sizes, np.mean(train_scores, axis=1), np.mean(test_scores, axis=1)


def model_equation(model, X_train, y_train, X_test, y_test, cible="Kt"):
    """Équation d'un modèle linéaire sous forme de texte, et son R² sur la base de test.

    >>> from sklearn.linear_model import LinearRegression
    >>> X = pd.DataFrame({"x": [1.0, 2.0, 3.0, 4.0]})
    >>> y = pd.Series([3.0, 5.0, 7.0, 9.0])
    >>> reg = LinearRegression().fit(X, y)
    >>> equation, r2 = model_equation(reg, X, y, X, y, cible="y")
    >>> equation, round(float(r2), 3)
    ('y = 1.000 + (2.00000 × x)', 1.0)
    """
    from sklearn.metrics import r2_score

    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)

    equation = f"{cible} = {model.intercept_:.3f}"

    for coef, name in zip(model.coef_, X_train.columns):
        equation += f" + ({coef:.5f} × {name})"

    return equation, r2


# Partie 5. Données prêtes pour les réseaux de neurones (chapitre 19)

def fenetres_glissantes(valeurs, pas, cible=None):
    """Découpe une série (temps, variables) en fenêtres de `pas` instants : tenseur (échantillons, pas, variables).

    Avec `cible`, rend aussi y : la cible lue à la fin de chaque fenêtre.

    >>> serie = np.arange(12.0).reshape(6, 2)
    >>> X = fenetres_glissantes(serie, 3)
    >>> X.shape
    (4, 3, 2)
    >>> X[1].tolist()
    [[2.0, 3.0], [4.0, 5.0], [6.0, 7.0]]
    >>> X, y = fenetres_glissantes(serie, 3, cible=np.arange(6))
    >>> y.tolist()
    [2, 3, 4, 5]
    """
    valeurs = np.asarray(valeurs)
    n = len(valeurs) - pas + 1
    if n <= 0:
        raise ValueError("série plus courte que la fenêtre")
    X = np.stack([valeurs[i:i + pas] for i in range(n)])
    if cible is None:
        return X
    return X, np.asarray(cible)[pas - 1:]


def normaliser_sans_fuite(X_train, X_test, axes=0):
    """Centre et réduit avec la moyenne et l'écart-type de l'apprentissage seul : rend X_train, X_test, moyenne, écart.

    `axes` désigne les axes qui portent les échantillons : 0 pour un tableau (lignes, variables), (0, 1) pour
    un tenseur de fenêtres (échantillons, pas de temps, variables) dont on veut une moyenne par variable.

    >>> a = np.array([[1.0], [2.0], [3.0]])
    >>> b = np.array([[4.0]])
    >>> a_n, b_n, moyenne, ecart_type = normaliser_sans_fuite(a, b)
    >>> round(float(a_n.mean()), 6), round(float(b_n[0, 0]), 3)
    (0.0, 2.449)
    >>> fenetres = np.arange(24.0).reshape(2, 3, 4)
    >>> normaliser_sans_fuite(fenetres, fenetres, axes=(0, 1))[2].shape
    (4,)
    """
    moyenne = X_train.mean(axis=axes)
    ecart = X_train.std(axis=axes)
    ecart = np.where(ecart == 0, 1.0, ecart)
    return (X_train - moyenne) / ecart, (X_test - moyenne) / ecart, moyenne, ecart


def _fonctions_publiques():
    """Les fonctions définies dans ce module, dans l'ordre du fichier."""
    import inspect

    return [nom for nom, obj in globals().items()
            if inspect.isfunction(obj) and obj.__module__ == __name__ and not nom.startswith("__")
            and nom != "_fonctions_publiques"]


if __name__ == "__main__":
    import doctest

    resultat = doctest.testmod()
    print(f"meca : {len(_fonctions_publiques())} fonctions, {resultat.failed} échec(s) sur {resultat.attempted} exemples")
    raise SystemExit(1 if resultat.failed else 0)
'''

`test_meca.py`

In [ ]:
TEXTE_TEST_MECA = r'''"""Tests unitaires du module meca (unittest, rejouables par pytest).

    python -m pytest test_meca.py -q      ou      python -m unittest test_meca -v

Trois familles : les exemples des docstrings (doctest), les fonctions de l'auteur (nombre et exemples),
et des valeurs de référence calculées par ailleurs (oracles exacts du calcul symbolique, jeux canoniques).
"""
import doctest
import inspect
import tempfile
import unittest
from pathlib import Path

import numpy as np
import pandas as pd
from numpy.testing import assert_allclose

import fiches
import meca
from oracles_sympy import confronter


class TestDoctest(unittest.TestCase):
    def test_exemples_des_docstrings(self):
        resultat = doctest.testmod(meca)
        self.assertEqual(resultat.failed, 0)
        self.assertGreaterEqual(resultat.attempted, 80)


class TestFonctionsAuteur(unittest.TestCase):
    def test_quinze_fonctions_de_l_auteur_presentes_et_testees(self):
        self.assertEqual(len(meca.FONCTIONS_AUTEUR), 15)
        for nom in meca.FONCTIONS_AUTEUR:
            fonction = getattr(meca, nom)
            self.assertTrue(inspect.isfunction(fonction), nom)
            self.assertIn(">>>", fonction.__doc__ or "", "pas d'exemple doctest : " + nom)

    def test_au_moins_dix_fonctions_vertes(self):
        # le chapitre « Du script à l'outil » existe si le module porte au moins 10 fonctions vertes
        finder = doctest.DocTestFinder()
        runner = doctest.DocTestRunner(verbose=False)
        vertes = 0
        for nom in meca._fonctions_publiques():
            for test in finder.find(getattr(meca, nom), nom, globs=vars(meca).copy()):
                if not test.examples:
                    continue
                resultat = runner.run(test, clear_globs=False)
                if resultat.failed == 0:
                    vertes += 1
        self.assertGreaterEqual(vertes, 10)


class TestCalculsContreOracles(unittest.TestCase):
    def test_kt_et_contrainte_de_la_plaque(self):
        kt = meca.kt_trou(10.0, 50.0)
        confronter("kt_plaque_02", kt)
        confronter("contrainte_max_MPa", kt * 20000.0 / ((50.0 - 10.0) * 5.0))

    def test_fleche_de_la_poutre(self):
        confronter("fleche_mm", meca.fleche_mm(2000.0, 1000.0, 20.0, 40.0, 210000.0))

    def test_arbre_par_dichotomie(self):
        d, tours = meca.dichotomie(lambda d: meca.tau_MPa(d, 500.0) - 60.0, 10.0, 100.0)
        confronter("arbre_mm", d)
        self.assertEqual(tours, 15)

    def test_garde_fous(self):
        with self.assertRaises(ValueError):
            meca.contrainte_MPa(1000.0, -1.0)
        with self.assertRaises(ValueError):
            meca.dichotomie(lambda x: x**2 + 1.0, 0.0, 1.0)
        self.assertTrue(np.isnan(meca.kt_trou(0.0, 50.0)))


class TestChaineDeCotes(unittest.TestCase):
    def test_ecart_type_du_jeu(self):
        cotes = [(100.0, 0.3, 1), (60.0, 0.15, -1), (38.0, 0.15, -1)]
        jeu = meca.jeu_monte_carlo(cotes, 200000, np.random.default_rng(1))
        attendu = np.sqrt(sum((t / 3)**2 for _, t, _ in cotes))
        assert_allclose(jeu.std(ddof=1), attendu, rtol=0.01)
        assert_allclose(jeu.mean(), 2.0, atol=0.002)


class TestPandas(unittest.TestCase):
    def test_horodatage_et_unite_de_la_fiche(self):
        d = pd.DataFrame({"time": [0.0, 1.0], "T": [1.0, 2.0]})
        jours = meca.indexer_horodatage(d, pd.Timestamp("2025-01-01"), unite="D")
        heures = meca.indexer_horodatage(d, pd.Timestamp("2025-01-01"), unite="h")
        self.assertEqual(str(jours.index[1]), "2025-01-02 00:00:00")
        self.assertEqual(str(heures.index[1]), "2025-01-01 01:00:00")

    def test_manquants_sur_le_suivi_de_production(self):
        suivi = fiches.lire_jeu("Suivi_Production_Machines")
        cellules, lignes = meca.cellules_et_lignes(suivi)
        self.assertEqual(cellules, 6043)
        self.assertEqual(lignes, 5462)
        self.assertLess(lignes, cellules)
        taux = meca.taux_manquants(suivi)
        self.assertTrue(taux.is_monotonic_decreasing)

    def test_bornes_physiques_du_jeu_rul(self):
        rul = fiches.lire_jeu("RUL")
        ecarts = meca.verifier_bornes(rul, {"RUL_jours": (0, 365), "masse_vibrante_kg": (0, 5000)})
        self.assertEqual(sum(ecarts.values()), 0)


class TestPretraitementAuteur(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        chemin = fiches.dossier_donnees() / "MecaIA-Plaque_Trouee_FEA.xlsx"
        cls.plaque = pd.read_excel(chemin)

    def test_taille_des_jeux_apprentissage_et_test(self):
        X_train, X_test, y_train, y_test = meca.preprocessing(self.plaque.copy())
        self.assertEqual(len(X_train) + len(X_test), 99)        # 1 % de 100 lignes retirées
        self.assertEqual(len(X_test), 20)
        self.assertEqual(X_train.shape[1], 1)

    def test_ordre_sans_fuite_laisse_le_test_intact(self):
        _, X_test_a, _, _ = meca.preprocessing(self.plaque.copy(), ordre="auteur")
        X_train_s, X_test_s, _, _ = meca.preprocessing(self.plaque.copy(), ordre="sans_fuite")
        self.assertEqual(len(X_test_s), 20)
        self.assertEqual(len(X_train_s), 79)
        self.assertEqual(len(X_test_a), 20)

    def test_kt_de_la_plaque_est_physique(self):
        out = meca.preprocessing_imputation(self.plaque.copy())
        self.assertTrue(out["Kt"].between(1.0, 10.0).all())

    def test_evaluation_et_equation(self):
        from sklearn.linear_model import LinearRegression
        X_train, X_test, y_train, y_test = meca.preprocessing(self.plaque.copy())
        modele = LinearRegression().fit(X_train, y_train)
        score = meca.evaluation_perf(modele, X_train, y_train, X_test, y_test, tol=0.05)
        self.assertTrue(0.0 <= score <= 100.0)
        equation, r2 = meca.model_equation(modele, X_train, y_train, X_test, y_test)
        self.assertTrue(equation.startswith("Kt = "))
        self.assertLessEqual(r2, 1.0)


class TestReseaux(unittest.TestCase):
    def test_fenetres_et_normalisation_sans_fuite(self):
        serie = np.arange(40.0).reshape(20, 2)
        X, y = meca.fenetres_glissantes(serie, 5, cible=np.arange(20))
        self.assertEqual(X.shape, (16, 5, 2))
        self.assertEqual(y[0], 4)
        a, b, moyenne, ecart = meca.normaliser_sans_fuite(serie[:15], serie[15:])
        assert_allclose(a.mean(axis=0), 0.0, atol=1e-12)
        assert_allclose(moyenne, serie[:15].mean(axis=0))      # la moyenne ignore le test

    def test_utilitaires_de_l_auteur(self):
        assert_allclose(meca.minmax_scale_01([0.0, 5.0, 10.0]), [0.0, 0.5, 1.0])
        assert_allclose(meca.safe_div(np.array([1.0]), np.array([0.0])), [0.0])
        m, b = meca.fit_line([0.0, 1.0, 2.0], [1.0, 3.0, 5.0])
        assert_allclose([m, b], [2.0, 1.0], atol=1e-9)
        self.assertEqual(meca.eq_label(m, b), "y = 2.00X + 1.00")


class TestFichiers(unittest.TestCase):
    def test_lecture_et_copie_de_securite(self):
        with tempfile.TemporaryDirectory() as dossier:
            chemin = Path(dossier) / "mesures.xlsx"
            pd.DataFrame({"x": [1, 2, 3]}).to_excel(chemin, index=False)
            data, df = meca.lire_classeur(chemin)
            df.loc[0, "x"] = 99
            self.assertEqual(int(data.loc[0, "x"]), 1)


if __name__ == "__main__":
    unittest.main()
'''

`oracles_sympy.py`

In [ ]:
TEXTE_ORACLES_SYMPY = r'''"""Valeurs exactes du calcul symbolique (chapitre 10), lues par les tests du module : une valeur, une unité,
une tolérance et le sens de chaque grandeur."""

ORACLES = {
    "kt_plaque_02": dict(
        valeur=2.512,
        unite='sans unité',
        tolerance=0.005,
        chapitres='3',
        sens='Kt de Heywood pour d/W = 0,2, sur la contrainte nette',
    ),
    "contrainte_max_MPa": dict(
        valeur=251.2,
        unite='MPa',
        tolerance=0.05,
        chapitres='3',
        sens='contrainte maximale de la plaque percée de 10 mm',
    ),
    "fleche_mm": dict(
        valeur=1.8601190476190477,
        unite='mm',
        tolerance=5e-05,
        chapitres='3, 10, 11',
        sens='flèche au milieu, poutre 20 x 40 mm, F = 2 kN, L = 1 m, acier',
    ),
    "arbre_mm": dict(
        valeur=34.881591197368635,
        unite='mm',
        tolerance=0.005,
        chapitres='3, 12',
        sens="diamètre de l'arbre : T = 500 N.m, tau = 60 MPa",
    ),
    "warren_fleche_mm": dict(
        valeur=-1.2499500278841598,
        unite='mm',
        tolerance=0.005,
        chapitres='11',
        sens='flèche du nœud chargé, axe y vers le haut (négative)',
    ),
    "warren_efforts_kN": dict(
        valeur=[6.666666666666667, 6.666666666666667, -13.333333333333334, -12.01850425154663, 12.01850425154663, 12.01850425154663, -12.01850425154663],
        unite='kN',
        tolerance=0.005,
        chapitres='11',
        sens='sept efforts normaux, + traction, ordre des barres du chapitre',
    ),
    "frequence_1_Hz": dict(
        valeur=66.78316718548766,
        unite='Hz',
        tolerance=0.005,
        chapitres='11, 16',
        sens='première fréquence propre, deux masses',
    ),
    "frequence_2_Hz": dict(
        valeur=138.49771874374207,
        unite='Hz',
        tolerance=0.05,
        chapitres='11, 16',
        sens='seconde fréquence propre, deux masses',
    ),
    "section_b_mm": dict(
        valeur=11.678449443205002,
        unite='mm',
        tolerance=0.005,
        chapitres='12',
        sens='largeur de la section rectangulaire de plus petite aire',
    ),
    "section_h_mm": dict(
        valeur=46.71379777282001,
        unite='mm',
        tolerance=0.005,
        chapitres='12',
        sens='hauteur de la section (h = 4 b, flèche L/500 atteinte)',
    ),
    "section_aire_mm2": dict(
        valeur=545.544725589981,
        unite='mm2',
        tolerance=0.05,
        chapitres='12',
        sens='aire minimale de la section',
    ),
    "lagrange_b_mm": dict(
        valeur=25.0,
        unite='mm',
        tolerance=0.001,
        chapitres='12',
        sens='Lagrange : b = D / 2 pour D = 50 mm',
    ),
    "lagrange_h_mm": dict(
        valeur=43.30127018922193,
        unite='mm',
        tolerance=0.001,
        chapitres='12',
        sens='Lagrange : h = D racine(3) / 2 pour D = 50 mm',
    ),
    "lagrange_fleche_mm": dict(
        valeur=1.1730291183535253,
        unite='mm',
        tolerance=5e-05,
        chapitres='12',
        sens='flèche minimale de la section inscrite dans le rond de 50 mm',
    ),
    "lagrange_multiplicateur": dict(
        valeur=-0.0009384232946828203,
        unite='1/mm',
        tolerance=1e-06,
        chapitres='12',
        sens='multiplicateur mu de grad f = mu grad g',
    ),
    "decrement_log": dict(
        valeur=0.31455270228880017,
        unite='sans unité',
        tolerance=5e-05,
        chapitres='13',
        sens='décrément logarithmique, m = 5 kg, k = 20 000 N/m, zeta = 0,05',
    ),
    "frequence_oscillateur_Hz": dict(
        valeur=10.065842420897408,
        unite='Hz',
        tolerance=0.005,
        chapitres='13',
        sens="fréquence propre non amortie de l'oscillateur",
    ),
    "norton_MPa": dict(
        valeur=295.3277414547264,
        unite='MPa',
        tolerance=0.005,
        chapitres='13',
        sens='contrainte résiduelle à 1 000 h, relaxation de Norton (n = 5)',
    ),
    "kt_035_exact": dict(
        valeur=2.274625,
        unite='sans unité',
        tolerance=5e-05,
        chapitres='10, 14',
        sens='Kt de Heywood pour d/W = 0,35',
    ),
    "kt_035_lineaire": dict(
        valeur=2.2795,
        unite='sans unité',
        tolerance=5e-05,
        chapitres='14',
        sens='Kt(0,35) lu linéairement entre d/W = 0,3 et 0,4',
    ),
    "kt_noeuds": dict(
        valeur=[2.729, 2.512, 2.343, 2.216, 2.125, 2.064, 2.027, 2.008],
        unite='sans unité',
        tolerance=5e-05,
        chapitres='14',
        sens='Kt de Heywood aux nœuds d/W = 0,1 ; 0,2 ; ... ; 0,8',
    ),
}


def valeur(nom):
    """Valeur exacte d'un oracle."""
    return ORACLES[nom]["valeur"]


def confronter(nom, calcule):
    """Compare un résultat numérique à l'oracle `nom` (assertion) et rend l'écart maximal."""
    fiche = ORACLES[nom]
    attendu = fiche["valeur"]
    couples = zip(calcule, attendu) if isinstance(attendu, list) else [(calcule, attendu)]
    delta = max(abs(float(c) - a) for c, a in couples)
    assert delta <= fiche["tolerance"], (
        f"{nom} : écart {delta:.3g} {fiche['unite']} au-delà de la tolérance {fiche['tolerance']}")
    return delta
'''

In [ ]:
import atexit
import shutil
import sys
import tempfile
from pathlib import Path

DOSSIER_MODULE = Path(tempfile.mkdtemp(prefix="meca_module_"))
atexit.register(shutil.rmtree, DOSSIER_MODULE, ignore_errors=True)       # effacé à la fermeture de Python
(DOSSIER_MODULE / "meca.py").write_text(TEXTE_MECA, encoding="utf-8")
(DOSSIER_MODULE / "test_meca.py").write_text(TEXTE_TEST_MECA, encoding="utf-8")
(DOSSIER_MODULE / "oracles_sympy.py").write_text(TEXTE_ORACLES_SYMPY, encoding="utf-8")

# Les tests lisent les jeux par un petit module `fiches` : ici, le dossier de la cellule de lecture.
TEXTE_FICHES = (
    "from pathlib import Path\n"
    "import pandas as pd\n\n"
    f"DONNEES = Path({str(DONNEES.resolve())!r})\n\n\n"
    "def dossier_donnees():\n"
    "    return DONNEES\n\n\n"
    "def lire_jeu(jeu):\n"
    "    return pd.read_excel(DONNEES / ('MecaIA-' + jeu + '.xlsx'))\n")
(DOSSIER_MODULE / "fiches.py").write_text(TEXTE_FICHES, encoding="utf-8")
sys.path.insert(0, str(DOSSIER_MODULE))
import meca

print("module importé :", meca.__name__, "version", meca.__version__)

### Préparation

In [ ]:
import runpy
import contextlib
import doctest
import importlib.metadata
import inspect
import io
import os
import re
import shutil
import tempfile
import unittest
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose

In [ ]:
import meca
pd.set_option("display.width", 100)

### Section 19.1 : un script devient un module.

In [ ]:
# notion: module
assert inspect.ismodule(meca) and Path(meca.__file__).suffix == ".py" and meca.__name__ == "meca"
noms_fonctions = meca._fonctions_publiques()
print(f"le module meca porte {len(noms_fonctions)} fonctions")
journal = io.StringIO()
trouveur = doctest.DocTestFinder()
verte = []                                                  # fonctions dont tous les exemples passent
sans_exemple = []

In [ ]:
for nom in noms_fonctions:
    tests = [t for t in trouveur.find(getattr(meca, nom), nom, globs=vars(meca).copy()) if t.examples]
    if not tests:
        sans_exemple.append(nom)
        continue
    executeur = doctest.DocTestRunner(verbose=False)
    with contextlib.redirect_stdout(journal):
        for t in tests:
            executeur.run(t, clear_globs=False)
    if executeur.failures == 0:
        verte.append(nom)

print(f"fonctions vertes par doctest : {len(verte)} ; sans exemple : {sans_exemple}")
flux_main = io.StringIO()

In [ ]:
with contextlib.redirect_stdout(flux_main):
    try:
        runpy.run_path(meca.__file__, run_name="__main__")
        code_retour = 0
    except SystemExit as sortie:
        code_retour = sortie.code or 0

ligne_finale = flux_main.getvalue().strip().splitlines()[-1]
print("python meca.py affiche :", ligne_finale)
trouve = re.fullmatch(r"meca : (\d+) fonctions, (\d+) échec\(s\) sur (\d+) exemples", ligne_finale)
assert code_retour == 0 and trouve, "le point d'entrée rejoue les exemples sans échec"
exemples_total = int(trouve.group(3))

In [ ]:
# notion: fonctions-auteur
assert (
    len(meca.FONCTIONS_AUTEUR) == 15 and all(hasattr(meca, nom) for nom in meca.FONCTIONS_AUTEUR)
)

for nom in ("preprocessing_imputation", "preprocessing_outliers", "preprocessing_Xy", "preprocessing"):
    print(f"{nom}{inspect.signature(getattr(meca, nom))}")

# un jeu construit dans le script : une plaque trouée de 200 mm de large, Kt de Heywood et bruit de 1 %
rng = np.random.default_rng(2026)
n = 100
d0 = np.linspace(10.0, 110.0, n)
section = 20.0 * (200.0 - d0)
contrainte_nominale = 1000.0 / section
kt_vrai = np.array([meca.kt_trou(d, 200.0) for d in d0])

In [ ]:
plaque = pd.DataFrame({"Unnamed: 0": range(n), "L": 400, "h": 200, "k": 20, "x0": 200, "y0": 100, "d0": d0,
                       "Material": "Steel-Generic",
                       "Stress/MPa": kt_vrai * contrainte_nominale * (1 + rng.normal(0, 0.01, n)),
                       "Disp/mm": 0.0005, "Msg": np.nan, "FEA_Runtime": rng.uniform(10, 15, n)})
reduit = meca.preprocessing_imputation(plaque)
sans_aberrants = meca.preprocessing_outliers(reduit)
X, y = meca.preprocessing_Xy(sans_aberrants)
X_train, X_test, y_train, y_test = meca.preprocessing(plaque)
print(f"plaque construite {plaque.shape} -> après imputation {reduit.shape}, sans aberrants {sans_aberrants.shape}")
print(f"X_train {X_train.shape}, X_test {X_test.shape}, y_train {y_train.shape}, y_test {y_test.shape}")
assert reduit.shape == (100, 2) and sans_aberrants.shape == (99, 2), "1 % de 100 lignes retiré"
assert (
    X.shape == (99, 1) and y.name == "Kt" and X_train.shape == (79, 1) and X_test.shape == (20, 1)
)

In [ ]:
assert (
    abs((reduit["Kt"] / kt_vrai).mean() - 1) < 0.01
), "le Kt reconstruit suit la formule de Heywood à 1 % près"
X_train_s, X_test_s, _, _ = meca.preprocessing(plaque, ordre="sans_fuite")
assert (
    X_test_s.shape == (20, 1) and X_train_s.shape == (79, 1)
), "le test garde ses 20 lignes, le retrait vise l'apprentissage"

## Exercice 19.1 : Combien de fonctions passent leurs exemples ? (15 min)

Comptez les fonctions du module `meca` dont tous les exemples de documentation passent, par `doctest.DocTestFinder` et `doctest.DocTestRunner`, et vérifiez que la liste `FONCTIONS_AUTEUR` en fait partie. Relevez les fonctions sans exemple.

**Résultat attendu.** 30 fonctions vertes sur 30, dont les 15 de l'auteur ; aucune fonction sans exemple ; 93 exemples rejoués.

**Seconde voie.** La commande `python meca.py`, lancée dans un terminal, affiche le même bilan : 30 fonctions, 0 échec.

**Indication.** Le `DocTestFinder`, par sa méthode `find`, rend les tests d'une fonction ; une fonction sans test n'a pas d'`examples`. Un `DocTestRunner` les joue, et `failures == 0` dit que la fonction est verte. On boucle sur la liste que rend `meca._fonctions_publiques()` et l'on range chaque nom dans la liste des vertes ou des sans exemple.

In [ ]:
# Votre code :
assert len(verte) >= 10, "au moins 10 fonctions vertes par doctest"
assert (
    len(verte) == len(noms_fonctions) and not sans_exemple
), "chaque fonction du module porte un exemple qui passe"
assert (
    all(nom in verte for nom in meca.FONCTIONS_AUTEUR)
), "les 15 fonctions de l'auteur sont vertes"
print("Exercice 19.1 : contrôles passés")

## Exercice 19.2 : Traiter un dossier de classeurs (20 min)

Dans un dossier neuf, rangez trois classeurs de votre choix parmi ceux du dépôt de données, plus un fichier abîmé du même genre que celui du chapitre. Écrivez le traitement par lot : pour chaque classeur, un rapport texte avec le nom du fichier, la taille du tableau, la part de cellules vides en pourcent et le résumé de sa dernière colonne par `meca._format_unique_values`. Une erreur ne doit pas arrêter le lot.

**Résultat attendu.** 4 fichiers dans le dossier ; 3 traités, 1 erreur isolée, 3 rapports écrits.

**Seconde voie.** Le nombre de rapports écrits est celui des fichiers traités, `len(list(rapports.glob("profil_*.txt")))`.

**Indication.** Pour chaque fichier listé par `glob` et pris dans l'ordre par `sorted`, un `try` lit le classeur avec `pd.read_excel`, calcule la part de cellules vides par `isna` et `mean`, appelle `meca._format_unique_values` sur sa dernière colonne, `tableau.iloc[:, -1]`, et écrit le rapport avec `write_text`. L'`except Exception` range le nom et le type de l'erreur ; la boucle continue.

Les contrôles de la cellule lisent les variables `contenu_rul`, `ecrits`, `erreurs`, `traites` : gardez ces noms.

In [ ]:
# Votre code :
assert (
    len(traites) == 3 and len(erreurs) == 1 and erreurs[0][0] == "MecaIA-Corrompu.xlsx"
), "3 traités, 1 erreur isolée"
assert (
    len(ecrits) == 3 and "240 lignes x 6 colonnes" in contenu_rul
), "un rapport par classeur lisible"
assert (
    dict(traites)["MecaIA-RUL.xlsx"] == (240, 6)
), "tailles de la fiche"
print("Exercice 19.2 : contrôles passés")

### Section 19.3 : tester, versionner, décrire.

In [ ]:
# notion: dataclass
@dataclass
class ParametresEvaluation:
    """Paramètres d'un traitement, rangés en un seul objet plutôt qu'en arguments dispersés."""
    tolerance: float = 0.05
    test_size: float = 0.2
    graine: int = 0

parametres = ParametresEvaluation(tolerance=0.02)
assert (
    parametres.tolerance == 0.02 and parametres.test_size == 0.2 and ParametresEvaluation() == ParametresEvaluation()
)
print("paramètres :", parametres)

In [ ]:
# notion: unittest
class TestMecaMinimal(unittest.TestCase):
    """Le plus petit fichier de tests d'un module : trois fonctions, trois valeurs connues."""

    def test_contrainte(self):
        assert_allclose(meca.contrainte_MPa(12000.0, 10.0), 152.79, atol=0.005)

    def test_fleche(self):
        assert_allclose(meca.fleche_mm(2000.0, 1000.0, 20.0, 40.0, 210000.0), 1.8601, atol=5e-5)

    def test_garde_fou(self):
        with self.assertRaises(ValueError):
            meca.contrainte_MPa(1000.0, 0.0)

In [ ]:
def lancer(suite):
    """Joue une suite de tests sans rien écrire à l'écran : rend le résultat."""
    flux = io.StringIO()
    return unittest.TextTestRunner(stream=flux, verbosity=0).run(suite)

petit = lancer(unittest.defaultTestLoader.loadTestsFromTestCase(TestMecaMinimal))
print(f"fichier de tests minimal : {petit.testsRun} tests, {len(petit.failures) + len(petit.errors)} échec")
assert petit.testsRun == 3 and petit.wasSuccessful()

In [ ]:
# notion: versions-outil
assert (
    re.fullmatch(r"\d+\.\d+\.\d+", meca.__version__)
), "version à trois nombres : majeure.mineure.correctif"
versions = {nom: importlib.metadata.version(nom) for nom in ("numpy", "pandas", "scipy", "scikit-learn", "matplotlib")}
gele = "\n".join(f"{nom}=={version}" for nom, version in versions.items())
print("versions figées :", gele.replace("\n", ", "))
journal_changements = f"## {meca.__version__}\n- fonctions de l'auteur reprises, exemples vérifiés par doctest\n"
assert gele.count("==") == 5 and journal_changements.startswith("## 1.0.0")

## Exercice 19.3 : Les tests unitaires du module (30 min)

Écrivez le fichier `test_meca.py` pour le module `meca`, en trois familles : les exemples de documentation rejoués par `doctest`, des valeurs de référence pour les calculs de résistance, une erreur attendue pour au moins un garde-fou. Visez au moins quinze tests, tous verts, et lancez-les avec `python -m unittest`.

**Résultat attendu.** Une suite d'au moins quinze tests sans échec ni erreur ; celle du module du livre en compte 18, contre 3 pour la suite minimale.

**Seconde voie.** `python -m pytest test_meca.py`, si `pytest` est installé, joue les mêmes tests et rend le même verdict.

**Indication.** Une classe par famille. Pour `doctest`, la fonction `testmod` appliquée à `meca` doit rendre `failed == 0` et plus de quatre-vingts exemples. Les valeurs de référence viennent de calculs faits par une autre voie : la flèche de $1{,}8601$ mm du chapitre 3, le diamètre d'arbre de la dichotomie. Les garde-fous se testent avec `assertRaises(ValueError)`. On compte les tests joués par `testsRun`.

Les contrôles de la cellule lisent la variable `complet` : gardez ce nom.

In [ ]:
# Votre code :
assert complet.testsRun >= 15 and complet.wasSuccessful(), "la suite du module est verte"
print("Exercice 19.3 : contrôles passés")

## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F19-1 : Arbre de décision (`sklearn.tree.DecisionTreeRegressor`)

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

X = np.array([[100, 2.0], [100, 8.0], [300, 2.0], [300, 8.0]])
y = np.array([40.5, 65.2, 55.0, 95.8])              # températures, °C
arbre = DecisionTreeRegressor(max_depth=2, random_state=0)
arbre.fit(X, y)

X_test = np.array([[200, 5.0]])                     # rad/s et kN
T_pred = arbre.predict(X_test)[0]
msg = f"T à 200 rad/s et 5 kN : {T_pred:.1f} °C"
print(msg.replace(".", ","))
print(np.isclose(arbre.score(X, y), 1.0))           # mémorisation parfaite

### Sections 19.4 à 19.7 : un premier modèle complet.

### Lecture des jeux de la partie modèle

Les jeux se cherchent d'abord dans le dépôt cloné, à deux dossiers au-dessus de ce notebook.

### Versions complémentaires

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [ ]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("joblib", version("joblib"))
print("matplotlib", version("matplotlib"))
print("numpy", version("numpy"))
print("pandas", version("pandas"))
print("scikit-learn", version("scikit-learn"))

### Fonctions du module du livre pour la partie modèle

Le code ci-dessous reprend, tel qu'il est écrit dans le livre, les fonctions du module employées dans ce notebook ; ce code est fourni tel quel.

In [ ]:
import numpy as np


def evaluation_perf(model, X_train, y_train, X_test, y_test, tol=0.05):
    """Pourcentage de prédictions de la base de test dans une tolérance relative `tol`.

    >>> from sklearn.linear_model import LinearRegression
    >>> X = pd.DataFrame({"x": [1.0, 2.0, 3.0, 4.0]})
    >>> y = pd.Series([2.0, 4.0, 6.0, 8.0])
    >>> reg = LinearRegression().fit(X, y)
    >>> float(evaluation_perf(reg, X, y, X, y, tol=0.01))
    100.0
    """
    y_pred = model.predict(X_test)
    rel_error = np.abs(y_pred - y_test) / np.abs(y_test)
    return 100 * np.mean(rel_error <= tol)


def evaluation_lc(model, X_train, y_train, tracer=True):
    """Courbe d'apprentissage (validation croisée à 5 plis, score R²) : rend les tailles et les deux scores moyens.

    >>> import matplotlib
    >>> matplotlib.use("Agg")
    >>> from sklearn.linear_model import LinearRegression
    >>> rng = np.random.default_rng(3)
    >>> X = pd.DataFrame({"x": rng.uniform(0, 10, 60)})
    >>> y = 2.0 * X["x"] + rng.normal(0, 0.5, 60)
    >>> tailles, entrainement, validation = evaluation_lc(LinearRegression(), X, y, tracer=False)
    >>> len(tailles), bool(validation[-1] > 0.9)
    (10, True)
    """
    from sklearn.model_selection import learning_curve

    train_sizes = np.linspace(0.1, 1.0, 10)
    sizes, train_scores, test_scores = learning_curve(
        model, X_train, y_train, train_sizes=train_sizes, cv=5, scoring="r2"
    )
    if tracer:
        import matplotlib.pyplot as plt

        plt.figure(figsize=(6, 4))
        plt.plot(sizes, np.mean(train_scores, axis=1), "o-", label="Train")
        plt.plot(sizes, np.mean(test_scores, axis=1), "o-", label="Validation")
        plt.xlabel("Taille de l'échantillon d'apprentissage")
        plt.ylabel("Score")
        plt.legend()
        plt.tight_layout()
    return sizes, np.mean(train_scores, axis=1), np.mean(test_scores, axis=1)


def model_equation(model, X_train, y_train, X_test, y_test, cible="Kt"):
    """Équation d'un modèle linéaire sous forme de texte, et son R² sur la base de test.

    >>> from sklearn.linear_model import LinearRegression
    >>> X = pd.DataFrame({"x": [1.0, 2.0, 3.0, 4.0]})
    >>> y = pd.Series([3.0, 5.0, 7.0, 9.0])
    >>> reg = LinearRegression().fit(X, y)
    >>> equation, r2 = model_equation(reg, X, y, X, y, cible="y")
    >>> equation, round(float(r2), 3)
    ('y = 1.000 + (2.00000 × x)', 1.0)
    """
    from sklearn.metrics import r2_score

    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)

    equation = f"{cible} = {model.intercept_:.3f}"

    for coef, name in zip(model.coef_, X_train.columns):
        equation += f" + ({coef:.5f} × {name})"

    return equation, r2


def fenetres_glissantes(valeurs, pas, cible=None):
    """Découpe une série (temps, variables) en fenêtres de `pas` instants : tenseur (échantillons, pas, variables).

    Avec `cible`, rend aussi y : la cible lue à la fin de chaque fenêtre.

    >>> serie = np.arange(12.0).reshape(6, 2)
    >>> X = fenetres_glissantes(serie, 3)
    >>> X.shape
    (4, 3, 2)
    >>> X[1].tolist()
    [[2.0, 3.0], [4.0, 5.0], [6.0, 7.0]]
    >>> X, y = fenetres_glissantes(serie, 3, cible=np.arange(6))
    >>> y.tolist()
    [2, 3, 4, 5]
    """
    valeurs = np.asarray(valeurs)
    n = len(valeurs) - pas + 1
    if n <= 0:
        raise ValueError("série plus courte que la fenêtre")
    X = np.stack([valeurs[i:i + pas] for i in range(n)])
    if cible is None:
        return X
    return X, np.asarray(cible)[pas - 1:]


def normaliser_sans_fuite(X_train, X_test, axes=0):
    """Centre et réduit avec la moyenne et l'écart-type de l'apprentissage seul : rend X_train, X_test, moyenne, écart.

    `axes` désigne les axes qui portent les échantillons : 0 pour un tableau (lignes, variables), (0, 1) pour
    un tenseur de fenêtres (échantillons, pas de temps, variables) dont on veut une moyenne par variable.

    >>> a = np.array([[1.0], [2.0], [3.0]])
    >>> b = np.array([[4.0]])
    >>> a_n, b_n, moyenne, ecart_type = normaliser_sans_fuite(a, b)
    >>> round(float(a_n.mean()), 6), round(float(b_n[0, 0]), 3)
    (0.0, 2.449)
    >>> fenetres = np.arange(24.0).reshape(2, 3, 4)
    >>> normaliser_sans_fuite(fenetres, fenetres, axes=(0, 1))[2].shape
    (4,)
    """
    moyenne = X_train.mean(axis=axes)
    ecart = X_train.std(axis=axes)
    ecart = np.where(ecart == 0, 1.0, ecart)
    return (X_train - moyenne) / ecart, (X_test - moyenne) / ecart, moyenne, ecart

### Préparation de la partie modèle

In [ ]:
import contextlib
import io
import tempfile
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
CHEMIN_RUL = DONNEES / "MecaIA-RUL.xlsx"
CHEMIN_FLOTTE = DONNEES / "MecaIA-TimeSeries_RUL.xlsx"

## Exercice 19.4 : Lire, nettoyer et apprendre une droite (15 min)

Lisez `MecaIA-RUL.xlsx`, retirez les lignes où une colonne utilisée manque, puis préparez $X$, la masse vibrante, en tableau à deux dimensions et $y$, la durée de vie, en vecteur. Affichez leurs formes et comptez les organes dont la durée de vie est inférieure à $120$ jours. Mettez un quart des lignes de côté (`random_state=42`), apprenez une droite sur l'apprentissage, puis relevez sa pente et son ordonnée à l'origine.

**Résultat attendu.** $X$ de forme $(240,\,1)$ et $y$ de forme $(240,)$ ; 56 organes à surveiller ; 180 lignes d'apprentissage et 60 de test ; pente de $-0{,}1221$ jour par kilogramme, ordonnée à l'origine de 369,3 jours.

**Seconde voie.** `np.polyfit` sur les lignes d'apprentissage redonne la même pente et la même ordonnée.

**Indication.** Deux crochets donnent un tableau à deux dimensions, un seul un vecteur : `df[["masse_vibrante_kg"]]` contre `df["RUL_jours"]`. La fonction `train_test_split` reçoit `test_size=0.25` et `random_state=42` et rend quatre morceaux ; `fit` apprend sur l'apprentissage, `coef_[0]` et `intercept_` donnent la pente et l'ordonnée.

In [ ]:
# Votre code :
assert X.shape == (240, 1), "X doit avoir 240 lignes et une colonne"
assert y.shape == (240,), "y doit être un vecteur de 240 valeurs"
# Votre code :
assert abs(pente - (-0.1221)) < 0.00005, "pente attendue : -0,1221 jour par kg"
assert abs(ordonnee - 369.3) < 0.05, "ordonnée attendue : 369,3 jours"
assert (len(df), 1 + 5) == (240, 6), "la fiche du jeu dit six colonnes"
# Votre code :
assert (
    abs(echelle.mean_[0] - Xa["masse_vibrante_kg"].mean()) < 1e-9 and set(arbre.classes_) == {"Faible", "Moyen", "Élevé"}
)
assert (
    abs(arbre.feature_importances_.sum() - 1.0) < 1e-12 and recherche.best_params_["max_depth"] in (1, 2, 3)
)
print("Exercice 19.4 : contrôles passés")

## Exercice 19.5 : Noter une droite sur la base de test (15 min)

Reprenez la droite de l'exercice 19.4 et notez-la : $R^2$ d'apprentissage et de test, RMSE et MAE sur la base de test, puis la part des prédictions à moins de $25\,\%$ de la valeur vraie. Comparez la RMSE à celle d'un modèle qui prédit toujours la moyenne de l'apprentissage.

**Résultat attendu.** $R^2$ de 0,285 à l'apprentissage et 0,358 au test ; RMSE de 72,4 jours, MAE de 55,2 jours ; 51,7 % des prédictions à moins de 25 %.

**Seconde voie.** Recalculez le $R^2$ de test à la main avec la formule : $1$ moins la somme des carrés des écarts du modèle sur la somme des carrés des écarts à la moyenne.

**Indication.** `droite.score` rend le $R^2$ ; `root_mean_squared_error` et `mean_absolute_error` reçoivent la vérité puis la prédiction. La moyenne de l'apprentissage, répétée autant de fois que la base de test a de lignes, donne la référence naïve : sa RMSE est supérieure à celle de la droite. La part dans la tolérance est la moyenne du masque `np.abs(prevu - vrai) / vrai <= 0.25`.

In [ ]:
# Votre code :
assert round(r2_test, 3) == 0.358, "R2 test attendu : 0,358"
assert abs(rmse_test - 72.4) < 0.05, "RMSE test attendue : 72,4 jours"
assert (
    abs(r2_score(yt, y_prevu) - r2_test) < 1e-12 and mae_test <= rmse_test
), "R2, MAE et RMSE concordent"
assert len(yt) == 60, "base de test attendue : 60 organes"
assert abs(r2_reduit - r2_test) < 1e-9, "la mise à l'échelle ne doit pas changer le R2"
# Votre code :
assert (
    abs(r2_equation - r2_test) < 1e-12 and equation.startswith("RUL = 369.3")
), "équation de l'auteur"
assert (
    0.0 <= dans_la_tolerance <= 100.0 and len(tailles) == 10 and tailles[-1] == int(0.8 * len(Xa))
)
# Votre code :
assert abs(sortie_1500 - (ordonnee + pente * 1500.0)) < 1e-9, "la sortie suit la droite apprise"
print("Exercice 19.5 : contrôles passés")

## Exercice 19.6 : Un script complet pour la durée de vie (20 min)

Écrivez un fichier qui lit le classeur, retire les lignes où une colonne utilisée manque, trace le nuage, apprend la droite sur trois quarts des lignes, note le $R^2$ de test, enregistre le modèle avec `joblib` puis le relit. Vérifiez que le modèle relu prédit comme le modèle enregistré et que le score est celui de la section 19.5.

**Résultat attendu.** Score de 0,358, identique à celui calculé pas à pas ; modèle relu qui prédit comme l'original ; fichier de 895 octets avec les versions du poste de rédaction.

**Seconde voie.** Lancez le fichier deux fois depuis un terminal avec `python` : le score doit être identique à chaque lancement.

**Indication.** Trois fonctions, une par groupe de gestes (lire et nettoyer, tracer, ajuster et évaluer), et une fonction `main` qui les enchaîne. `joblib.dump(modele, nom)` écrit le fichier, `joblib.load(nom)` le relit ; `np.allclose` compare les prédictions des deux modèles. La taille du fichier dépend des versions de scikit-learn et de joblib.

Les contrôles de la cellule lisent les variables `modele_complet`, `relu`, `score_complet` : gardez ces noms.

In [ ]:
# Votre code :
assert abs(score_complet - r2_test) < 1e-12, "le script complet retrouve le score pas à pas"
assert (
    np.allclose(relu.predict(Xt), modele_complet.predict(Xt))
), "le modèle relu prédit comme le modèle enregistré"
# Votre code :
print("Exercice 19.6 : contrôles passés")

## Exercice 19.7 : Des fenêtres et des images pour un réseau (défi, 35 min)

Sur la flotte, construisez le tenseur de fenêtres de $30$ relevés des cinq capteurs, avec la durée de vie restante au dernier relevé, sans jamais franchir deux machines. Apprenez sur les machines 1 à 80, testez sur les suivantes, et centrez et réduisez chaque capteur avec la moyenne et l'écart-type de l'apprentissage seul. Préparez ensuite les images $8 \times 8$ de `load_digits` : forme d'origine, à plat, avec canal, puis mise à l'échelle entre $0$ et $1$ image par image.

**Résultat attendu.** Tenseur de 15 955 fenêtres de 30 pas et 5 capteurs ; 12 393 fenêtres d'apprentissage et 3 562 de test ; apprentissage centré, test non centré ; 1 797 images de $8\times 8$, à plat 64 colonnes, valeurs de $0$ à 16, puis de $0$ à $1$ après la mise à l'échelle.

**Seconde voie.** Le nombre de fenêtres est le nombre de relevés moins $29$ par machine ; le nombre de colonnes à plat est $8\times 8$.

**Indication.** Pour chaque machine, `groupby("ID_Machine")` isole le suivi, `fenetres_glissantes(valeurs, 30, cible=...)` rend les fenêtres et leurs cibles, et `np.concatenate` les assemble. Les moyennes et les écarts-types sont pris sur l'apprentissage avec `axis=(0, 1)`, puis appliqués aux deux lots. Pour les images, `reshape(n, -1)` aplatit, `images[..., np.newaxis]` ajoute le canal, et $(x - \min)/(\max - \min)$ par image donne l'échelle unité.

In [ ]:
# Votre code :
assert (
    flotte.shape == (18855, 10)
), "la fiche du jeu dit la même taille"
assert (
    tenseur.shape == (len(flotte) - 100 * (pas - 1), pas, 5) == (15955, 30, 5)
), "15 955 fenêtres de 30 x 5"
assert vie_restante.shape == (15955,) and groupes.shape == (15955,)
# Votre code :
assert (
    np.abs(X_app_n.mean(axis=(0, 1))).max() < 1e-9 and np.abs(X_app_n.std(axis=(0, 1)) - 1).max() < 1e-9
)
assert (
    np.abs(moyenne_test_n).max() > 1e-3
), "le test n'est pas centré : il ne participe pas aux statistiques"
# Votre code :
assert (
    np.allclose(X_a_m, X_app_n) and np.allclose(moy_m, moyenne_app)
), "la fonction du module refait ce calcul"
# Votre code :
assert (
    images.shape == (1797, 8, 8) and a_plat.shape == (1797, 64) and avec_canal.shape == (1797, 8, 8, 1)
)
assert (
    images_01.min() == 0.0 and images_01.max() == 1.0 and np.array_equal(a_plat[0], images[0].ravel())
)
print("Exercice 19.7 : contrôles passés")

## Le score de référence

Le défi du chapitre note la droite de l'exercice 19.5 sur la base de test : un quart des lignes, mis de côté par `train_test_split` avec la graine 42. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

positions = ",".join(str(i) for i in sorted(Xt.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.3f" % r2_test)